In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

In [0]:
%pip install kaggle

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

In [0]:
%restart_python

In [0]:
df_n = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")

In [0]:
df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")

In [0]:
# Load October data
df_oct = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv",
    header=True,
    inferSchema=True
)

# Load November data
df_nov = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv",
    header=True,
    inferSchema=True
)


In [0]:
df_oct.show(2, truncate=False)
df_nov.show(2, truncate=False)

In [0]:
events = df_oct.unionByName(df_nov)

**✅ Select required columns**

In [0]:
events.select("event_type", "product_id", "price").show(10)

**Filter high-value events**

In [0]:
events.filter(events.price > 100).count()

**Group by event type**

In [0]:
events.groupBy("event_type").count().show()

**Top brands by activity**

In [0]:
top_brands = (
    events
    .groupBy("brand")
    .count()
    .orderBy("count", ascending=False)
    .limit(5)
)

top_brands.show()


In [0]:
df_oct = spark.read.csv(
    "dbfs:/user/hive/warehouse/2019-Oct.csv",
    header=True,
    inferSchema=True
)

df_nov = spark.read.csv(
    "dbfs:/user/hive/warehouse/2019-Nov.csv",
    header=True,
    inferSchema=True
)


In [0]:
events = df_oct.unionByName(df_nov)


In [0]:
products = (
    events
    .select("product_id", "brand", "price")
    .dropDuplicates(["product_id"])
)

In [0]:
users = (
    events
    .select("user_id")
    .dropDuplicates()
)


In [0]:
events

In [0]:
df_oct = spark.read.csv("dbfs:/user/hive/warehouse/2019-Oct.csv", header=True, inferSchema=True)
df_nov = spark.read.csv("dbfs:/user/hive/warehouse/2019-Nov.csv", header=True, inferSchema=True)

events = df_oct.unionByName(df_nov)


In [0]:
events = df_oct.unionByName(df_nov)


In [0]:
from pyspark.sql import functions as F

revenue = (
    events
    .filter(F.col("event_type") == "purchase")
    .groupBy("product_id", "brand")
    .agg(F.sum("price").alias("revenue"))
    .orderBy(F.desc("revenue"))
    .limit(5)
)

revenue.show(truncate=False)


In [0]:
%restart_python


In [0]:
df_oct = spark.read.csv(
    "dbfs:/user/hive/warehouse/2019-Oct.csv",
    header=True,
    inferSchema=True
)

df_nov = spark.read.csv(
    "dbfs:/user/hive/warehouse/2019-Nov.csv",
    header=True,
    inferSchema=True
)

events = df_oct.unionByName(df_nov)


In [0]:
events.show(5, truncate=False)


In [0]:
dbutils.fs.ls("dbfs:/Workspace/")


In [0]:
dbutils.fs.ls("dbfs:/Workspace/Users/")


In [0]:
from pyspark.sql.window import Window

window_spec = (
    Window
    .partitionBy("user_id")
    .orderBy(F.to_timestamp("event_time"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

events_with_cumulative = (
    events
    .filter(F.col("event_type") == "purchase")
    .withColumn("cumulative_events", F.count("*").over(window_spec))
)

events_with_cumulative.select(
    "user_id", "event_time", "cumulative_events"
).show(10, truncate=False)


In [0]:
from pyspark.sql.functions import when

events_features = events.withColumn(
    "high_value_purchase",
    when(col("price") > 1000, 1).otherwise(0)
)


In [0]:
from pyspark.sql.functions import hour

events_features = events_features.withColumn(
    "event_hour",
    hour(to_timestamp("event_time"))
)


In [0]:
events_features = events_features.withColumn(
    "revenue",
    when(col("event_type") == "purchase", col("price")).otherwise(0)
)